# Практикум 5. Многомерные методы

**Курс «Анализ больших данных»**

---

### Как работать с этим блокнотом

1. Выполняйте ячейки **по порядку**: `Shift + Enter`.
2. Подойдёт Jupyter (Anaconda или python.org + pip) или Google Colab — см. инструкцию в информационном
   разделе курса. Нужны numpy, pandas, scipy, matplotlib; sklearn и statsmodels — по желанию (есть в Colab).
3. 📘 — разбор: запустите и посмотрите результат. ✏️ — задание: допишите код вместо `...` или `# ваш код`, затем запустите
   ячейку проверки с `assert` — если ошибок нет, решение верно.
   💡 — решение под треугольником, открывайте после честной попытки.
4. Перед сдачей: **Kernel → Restart & Run All**.

### Что будет к концу блокнота

Вы найдёте главные компоненты характеристик средств размещения «КубаньТура» и дадите им смысл,
сгруппируете признаки факторным анализом, разобьёте объекты на кластеры двумя методами и опишете сегменты, научите
классификатор узнавать тип размещения и сегментируете 60 000 бронирований мини-пакетным k-средних — как на больших
данных. Каждое задание проверяется ячейкой с `assert`.

In [ ]:
import json, os, sys, time, random
from datetime import datetime, timedelta, timezone
from pprint import pprint
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 140)
BLUE, ORANGE = "#34519E", "#CF7A2A"      # цвета серий на графиках
NAVY, GREY = "#1E2761", "#9AA3B5"
IN_COLAB = "google.colab" in sys.modules
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| Colab" if IN_COLAB else "| локально")

Данные — кейс **«КубаньТур»** (условные; первая ячейка работает ≈ 3 с). Запустите её.

In [ ]:
# Сквозной кейс курса «Анализ больших данных» — «КубаньТур»: условная платформа бронирования
# средств размещения Краснодарского края. Все отели, туристы и бронирования вымышлены; генерация детерминирована
# (seed 2026), у всех получаются одинаковые числа. Только стандартная библиотека.
#   HOTELS          — 240 средств размещения: город, тип, звёзды, номера, персонал, расстояние до моря, удобства,
#                     рейтинг, базовая цена номера в сутки (высокий сезон)
#   bookings(n)     — ЛЕНИВЫЙ генератор n бронирований (словари) за 01.01.2025–30.09.2026
#   BOOKINGS        — 60 000 бронирований (список словарей) — основной набор для pandas
#   booking_lines(n)— ленивый генератор строк CSV «id;дата;отель;город;канал;ночей;гостей;сумма;отмена»
#   RAW_BOOKINGS    — 20 000 строк «сырой» выгрузки с дефектами (см. RAW_DEFECTS) — для разведочного анализа
import math as _m
import random as _random
from datetime import date as _date, timedelta as _td

_SEED = 2026
_TITLE = "КубаньТур"
_R = _random.Random(_SEED)
_D0, _D1 = _date(2025, 1, 1), _date(2026, 9, 30)

_CITIES = [  # город, доля отелей, у моря?, эффект цены (лог), «горный»?
    ("Сочи", 0.28, True, 0.30, False), ("Анапа", 0.17, True, 0.00, False), ("Геленджик", 0.14, True, 0.15, False),
    ("Краснодар", 0.12, False, 0.05, False), ("Красная Поляна", 0.09, False, 0.35, True),
    ("Туапсе", 0.08, True, -0.10, False), ("Ейск", 0.06, True, -0.25, False), ("Абрау-Дюрсо", 0.06, True, 0.20, False)]
_TYPES = [  # тип, доля, звёзды (мин, макс), номера (мин, макс), персонал на номер, эффект цены (лог)
    ("отель", 0.40, 2, 5, 40, 400, 0.45, 0.10), ("гостевой дом", 0.25, 0, 2, 5, 25, 0.15, -0.15),
    ("апартаменты", 0.15, 0, 3, 20, 120, 0.12, 0.05), ("санаторий", 0.10, 3, 4, 100, 500, 0.80, 0.20),
    ("хостел", 0.10, 0, 0, 10, 40, 0.10, -0.80)]
_WORDS = ["Бриз", "Волна", "Магнолия", "Кипарис", "Лазурь", "Прибой", "Пальма", "Чайка", "Маяк", "Риф", "Парус",
          "Гранат", "Лаванда", "Олива", "Горизонт", "Альпика", "Эдельвейс", "Кубань", "Таврия", "Фрегат", "Аврора",
          "Ривьера", "Жемчуг", "Солнце", "Ласточка", "Каштан", "Платан", "Янтарь", "Нептун", "Медуза"]
_PREFIX = {"отель": "Отель", "гостевой дом": "Гостевой дом", "апартаменты": "Апартаменты",
           "санаторий": "Санаторий", "хостел": "Хостел"}


def _pick(r, items, weights):
    return r.choices(items, weights=weights, k=1)[0]


HOTELS = []
for _i in range(1, 241):
    _c = _pick(_R, _CITIES, [c[1] for c in _CITIES])
    _t = _pick(_R, _TYPES, [t[1] for t in _TYPES])
    _stars = _R.randint(_t[2], _t[3])
    _rooms = _R.randint(_t[4], _t[5])
    _staff = max(1, round(_rooms * _t[6] * _R.lognormvariate(0, 0.15)))
    if _c[2]:
        _dist = round(min(4.0, _R.lognormvariate(_m.log(0.6), 0.8)), 2)
    elif _c[4]:
        _dist = round(40 + _R.uniform(-3, 3), 1)
    else:
        _dist = round(_R.uniform(115, 150), 1)
    _pool = _R.random() < 0.10 + 0.15 * _stars
    _spa = _R.random() < (0.05 + 0.12 * _stars + (0.4 if _t[0] == "санаторий" else 0))
    _breakfast = _R.random() < 0.3 + 0.12 * _stars
    _parking = _R.random() < 0.55 + (0.2 if not _c[2] else 0)
    _area = round(16 + 4.2 * _stars + _R.gauss(0, 3) + (12 if _t[0] == "апартаменты" else 0), 1)
    _rating = round(min(9.9, max(5.0, 6.4 + 0.45 * _stars - 0.25 * _m.log(_dist + 0.1) * (1 if _c[2] else 0)
                                + 0.3 * _spa + _R.gauss(0, 0.45))), 1)
    _lp = (_m.log(1600) + 0.33 * _stars + 0.22 * _pool + 0.18 * _spa + 0.10 * _breakfast + 0.012 * (_area - 25)
           - (0.12 * _m.log(_dist + 0.1) if _c[2] else 0) + _c[3] + _t[7] + 0.08 * (_rating - 8) + _R.gauss(0, 0.12))
    HOTELS.append({"hotel_id": f"H{_i:03d}", "name": f"{_PREFIX[_t[0]]} «{_R.choice(_WORDS)}»", "city": _c[0],
                   "type": _t[0], "stars": _stars, "rooms": _rooms, "staff": _staff, "dist_sea_km": _dist,
                   "pool": _pool, "spa": _spa, "breakfast": _breakfast, "parking": _parking, "room_area": _area,
                   "rating": _rating, "price_base": int(round(_m.exp(_lp), -1))})
_HOTEL = {h["hotel_id"]: h for h in HOTELS}
_CITY = {c[0]: c for c in _CITIES}
_REGIONS = [("Москва", 0.24), ("Краснодарский край", 0.18), ("Московская обл.", 0.10), ("Санкт-Петербург", 0.09),
            ("Ростовская обл.", 0.08), ("Самарская обл.", 0.05), ("Свердловская обл.", 0.05),
            ("Республика Татарстан", 0.05), ("Воронежская обл.", 0.04), ("другие", 0.12)]
_CHANNELS = [("сайт", 0.25, 1.00), ("приложение", 0.30, 0.97), ("агрегатор", 0.35, 1.06), ("офис", 0.10, 1.03)]
_AGES = [("18–24", 0.12), ("25–34", 0.30), ("35–44", 0.27), ("45–54", 0.16), ("55+", 0.15)]


def _season(month: int, city: tuple) -> float:
    """Сезонный множитель цены и спроса: море — пик в июле–августе, горы — зимой и летом, Краснодар — ровно."""
    if city[4]:
        return [1.25, 1.30, 1.05, 0.70, 0.75, 0.90, 1.05, 1.10, 0.85, 0.70, 0.75, 1.20][month - 1]
    if not city[2]:
        return [0.85, 0.85, 0.95, 1.00, 1.05, 1.05, 1.00, 1.00, 1.05, 1.05, 0.95, 1.00][month - 1]
    return [0.45, 0.45, 0.50, 0.60, 0.80, 1.05, 1.35, 1.40, 1.00, 0.65, 0.50, 0.55][month - 1]


def bookings(n: int, seed: int = _SEED):
    """Ленивый генератор n бронирований (словари). Одинаковый seed — одинаковые данные."""
    r = _random.Random(seed + 1)
    span = (_D1 - _D0).days
    hotel_w = [h["rooms"] ** 0.7 * (1 + (h["rating"] - 7) / 3) for h in HOTELS]
    for k in range(1, n + 1):
        h = r.choices(HOTELS, weights=hotel_w, k=1)[0]
        c = _CITY[h["city"]]
        while True:                                             # дата заезда с учётом сезонного спроса
            d = _D0 + _td(days=r.randrange(span + 1))
            if r.random() < _season(d.month, c) / 1.45:
                break
        ch = _pick(r, _CHANNELS, [x[1] for x in _CHANNELS])
        lead = min(240, int(r.expovariate(1 / (35 if ch[0] != "офис" else 55))))
        nights = max(1, min(21, int(r.lognormvariate(_m.log(4.2 if c[2] else (3.2 if c[4] else 1.8)), 0.55))))
        guests = r.choices([1, 2, 3, 4], weights=[18, 50, 18, 14], k=1)[0]
        if h["type"] == "хостел":
            guests = 1
        wknd = 1.08 if d.weekday() >= 4 else 1.0
        disc = 0.92 if lead > 60 else 1.0
        price = h["price_base"] * _season(d.month, c) * wknd * disc * ch[2] * r.lognormvariate(0, 0.08)
        price *= 1 + 0.25 * (guests - 2) * (h["type"] in ("апартаменты", "санаторий"))
        price = int(round(max(500, price), -1))
        p_cancel = 0.06 + 0.07 * (ch[0] == "агрегатор") + 0.0006 * lead
        cancelled = r.random() < p_cancel
        score = None
        if not cancelled and r.random() < 0.35:
            score = max(1, min(10, round(h["rating"] + r.gauss(0, 1.1) - (0.4 if guests >= 3 else 0))))
        yield {"booking_id": f"B{k:07d}", "created": (d - _td(days=lead)).isoformat(), "checkin": d.isoformat(),
               "hotel_id": h["hotel_id"], "city": h["city"], "type": h["type"], "channel": ch[0],
               "region": _pick(r, [x[0] for x in _REGIONS], [x[1] for x in _REGIONS]),
               "age_group": _pick(r, [x[0] for x in _AGES], [x[1] for x in _AGES]),
               "lead_days": lead, "nights": nights, "guests": guests, "price_night": price,
               "amount": price * nights, "cancelled": cancelled, "review_score": score}


BOOKINGS = list(bookings(60_000))


def booking_lines(n: int, seed: int = _SEED):
    """Ленивый генератор строк журнала «id;дата_заезда;отель;город;канал;ночей;гостей;сумма;отмена» (без заголовка)."""
    for b in bookings(n, seed):
        yield (f"{b['booking_id']};{b['checkin']};{b['hotel_id']};{b['city']};{b['channel']};{b['nights']};"
               f"{b['guests']};{b['amount']};{int(b['cancelled'])}")


# «сырая» выгрузка: те же поля, но с типичными дефектами реальных данных
_RR = _random.Random(_SEED + 7)
RAW_BOOKINGS, RAW_DEFECTS = [], {"формат даты дд.мм.гггг": 0, "город с ошибкой написания": 0,
                                 "сумма строкой «… руб.»": 0, "пропуск ночей": 0, "пропуск канала": 0,
                                 "выброс цены (×100)": 0, "выброс ночей (365)": 0, "отрицательные гости": 0,
                                 "полный дубль": 0}
for _b in BOOKINGS[:20_000]:
    _row = {k: _b[k] for k in ("booking_id", "checkin", "hotel_id", "city", "channel", "nights", "guests",
                               "price_night", "cancelled")}
    _u = _RR.random()
    if _u < 0.06:
        _row["checkin"] = _date.fromisoformat(_row["checkin"]).strftime("%d.%m.%Y"); RAW_DEFECTS["формат даты дд.мм.гггг"] += 1
    elif _u < 0.10:
        _row["city"] = _RR.choice([_row["city"].lower(), _row["city"] + " ", "г. " + _row["city"]]); RAW_DEFECTS["город с ошибкой написания"] += 1
    elif _u < 0.13:
        _row["price_night"] = f"{_row['price_night']:,} руб.".replace(",", " "); RAW_DEFECTS["сумма строкой «… руб.»"] += 1
    elif _u < 0.155:
        _row["nights"] = None; RAW_DEFECTS["пропуск ночей"] += 1
    elif _u < 0.175:
        _row["channel"] = None; RAW_DEFECTS["пропуск канала"] += 1
    elif _u < 0.180:
        _row["price_night"] = _row["price_night"] * 100; RAW_DEFECTS["выброс цены (×100)"] += 1
    elif _u < 0.184:
        _row["nights"] = 365; RAW_DEFECTS["выброс ночей (365)"] += 1
    elif _u < 0.187:
        _row["guests"] = -_row["guests"]; RAW_DEFECTS["отрицательные гости"] += 1
    RAW_BOOKINGS.append(_row)
    if _RR.random() < 0.015:
        RAW_BOOKINGS.append(dict(_row)); RAW_DEFECTS["полный дубль"] += 1

print(f"КубаньТур: отелей {len(HOTELS)}, бронирований {len(BOOKINGS)}, строк сырой выгрузки {len(RAW_BOOKINGS)}")

## 1. Данные и стандартизация 📘

Десять характеристик в разных единицах — приводим к z-оценкам.

In [ ]:
# Данные: 10 характеристик объектов, стандартизация
from scipy import stats
from scipy.cluster.vq import kmeans2
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import cdist
from scipy.special import comb

h = pd.DataFrame(HOTELS)
h["ldist"] = np.log(h.dist_sea_km + 0.1)
F = ["stars", "rooms", "staff", "room_area", "rating", "ldist", "pool", "spa", "breakfast", "parking"]
X = h[F].astype(float)
Z = ((X - X.mean()) / X.std()).to_numpy()
print(Z.shape, Z.mean(axis=0).round(12).max(), Z.std(axis=0, ddof=1).round(6).tolist()[:3])

## 2. Главные компоненты 📘

Собственные значения и векторы корреляционной матрицы, нагрузки, проекция объектов.

In [ ]:
# Главные компоненты: собственные значения и векторы корреляционной матрицы, нагрузки
R = np.corrcoef(Z.T)
val, vec = np.linalg.eigh(R)
order = np.argsort(val)[::-1]
val, vec = val[order], vec[:, order]
vec = vec * np.sign(vec[np.abs(vec).argmax(axis=0), np.arange(len(F))])
print("λ:", val.round(2).tolist(), "| накоплено:", (val.cumsum() / val.sum()).round(3).tolist())
load = pd.DataFrame(vec[:, :3] * np.sqrt(val[:3]), index=F, columns=["ГК1", "ГК2", "ГК3"])
scores = Z @ vec
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].bar(range(1, 11), val, color=BLUE); ax[0].axhline(1, color=GREY, ls="--"); ax[0].set_title("каменистая осыпь")
for t, g in h.groupby("type"):
    ax[1].scatter(scores[g.index, 0], scores[g.index, 1], s=10, label=t)
ax[1].set_xlabel("ГК1"); ax[1].set_ylabel("ГК2"); ax[1].legend(fontsize=8)
plt.show()
load.round(2)

## 3. Факторный анализ 📘

Вращение варимакс группирует признаки по факторам.

In [ ]:
# Факторный анализ: нагрузки главных компонент + вращение варимакс
def varimax(L, iters=200, tol=1e-10):
    p, k = L.shape
    rot, d = np.eye(k), 0.0
    for _ in range(iters):
        Lr = L @ rot
        u, s, vt = np.linalg.svd(L.T @ (Lr ** 3 - Lr @ np.diag((Lr ** 2).sum(axis=0)) / p))
        rot = u @ vt
        if s.sum() < d * (1 + tol):
            break
        d = s.sum()
    return L @ rot

fa = pd.DataFrame(varimax(vec[:, :3] * np.sqrt(val[:3])), index=F, columns=["Ф1", "Ф2", "Ф3"])
fa["общность"] = (fa ** 2).sum(axis=1)
fa.round(2)

## 4. k-средних 📘

Несколько запусков, выбор k, профили кластеров.

In [ ]:
# k-средних: несколько запусков, локоть и силуэт, профили
def kmeans(X, k, runs=10):
    best = None
    for s in range(runs):
        c, lab = kmeans2(X, k, minit="++", seed=s)
        inertia = ((X - c[lab]) ** 2).sum()
        if best is None or inertia < best[2]:
            best = (c, lab, inertia)
    return best

D = cdist(Z, Z)

def silhouette(lab):
    s = np.zeros(len(lab))
    for i in range(len(lab)):
        own = lab == lab[i]
        a = D[i, own].sum() / max(own.sum() - 1, 1)
        b = min(D[i, lab == c].mean() for c in np.unique(lab) if c != lab[i])
        s[i] = (b - a) / max(a, b)
    return s.mean()

for k in range(2, 8):
    _, lab, inertia = kmeans(Z, k)
    print(k, round(inertia, 1), round(silhouette(lab), 3))
_, lab5, _ = kmeans(Z, 5)
h.assign(кластер=lab5).groupby("кластер")[F + ["price_base"]].mean().round(2)

## 5. Иерархическая кластеризация 📘

Метод Уорда, дендрограмма, ARI.

In [ ]:
# Иерархическая кластеризация (Уорд), дендрограмма, сравнение разбиений (ARI)
def ari(a, b):
    ct = pd.crosstab(np.asarray(a), np.asarray(b)).to_numpy()
    sc, sa, sb = comb(ct, 2).sum(), comb(ct.sum(1), 2).sum(), comb(ct.sum(0), 2).sum()
    exp = sa * sb / comb(ct.sum(), 2)
    return (sc - exp) / ((sa + sb) / 2 - exp)

link = linkage(Z, method="ward")
lab_w = fcluster(link, 5, criterion="maxclust") - 1
fig, ax = plt.subplots(figsize=(12, 3)); dendrogram(link, ax=ax, no_labels=True); plt.show()
print("ARI: k-средних и Уорд", round(ari(lab5, lab_w), 3), "| k-средних и тип", round(ari(lab5, h.type), 3))

## 6. Классификация с учителем 📘

LDA и k ближайших соседей; точность на кросс-валидации.

In [ ]:
# Линейный дискриминантный анализ и k ближайших соседей, кросс-валидация
def lda_fit(X, y):
    cls = np.unique(y)
    mu = np.array([X[y == c].mean(axis=0) for c in cls])
    S = sum((X[y == c] - mu[i]).T @ (X[y == c] - mu[i]) for i, c in enumerate(cls)) / (len(y) - len(cls))
    return cls, mu, np.linalg.inv(S), np.array([(y == c).mean() for c in cls])

def lda_predict(model, X):
    cls, mu, Si, pri = model
    d = X @ Si @ mu.T - 0.5 * np.einsum("kj,jl,kl->k", mu, Si, mu) + np.log(pri)
    return cls[d.argmax(axis=1)]

def knn_predict(Xtr, ytr, X, k=5):
    near = np.argsort(cdist(X, Xtr), axis=1)[:, :k]
    return np.array([pd.Series(ytr[r]).mode()[0] for r in near])

def cv_accuracy(fit_predict, X, y, k=5, seed=1):
    idx = np.random.default_rng(seed).permutation(len(y)); pred = np.empty(len(y), dtype=object)
    for part in np.array_split(idx, k):
        tr = np.setdiff1d(idx, part)
        pred[part] = fit_predict(X[tr], y[tr], X[part])
    return (pred == y).mean(), pred

y = h.type.to_numpy()
acc_lda, pred_lda = cv_accuracy(lambda a, b, c: lda_predict(lda_fit(a, b), c), Z, y)
print("LDA:", round(acc_lda, 3), "| 5 соседей:", round(cv_accuracy(knn_predict, Z, y)[0], 3))
pd.crosstab(pd.Series(y, name="факт"), pd.Series(pred_lda, name="прогноз"))

## 7. Большие данные 📘

Ковариационная матрица по блокам.

In [ ]:
# Главные компоненты на больших данных: n, Σx, Σxxᵀ по блокам
def feats(rows):
    d = pd.DataFrame(rows)
    return np.column_stack([np.log(d.price_night), np.log(d.nights), np.log(d.amount), d.guests, d.lead_days])

n_, s1, s2 = 0, np.zeros(5), np.zeros((5, 5))
for i in range(0, len(BOOKINGS), 10_000):
    xb = feats(BOOKINGS[i:i + 10_000]); n_ += len(xb); s1 += xb.sum(0); s2 += xb.T @ xb
mean = s1 / n_
cov = (s2 - n_ * np.outer(mean, mean)) / (n_ - 1)
corr = cov / np.sqrt(np.outer(np.diag(cov), np.diag(cov)))
print("λ:", (np.linalg.eigvalsh(corr)[::-1].round(4) + 0.0).tolist(), "— нулевое: ln суммы = ln цены + ln ночей")

---

# Задания ✏️

Пишите код вместо `...` / `# ваш код`, затем запускайте **ячейку проверки** под заданием.

## Задание 1. Главные компоненты через SVD

Связь SVD и собственных значений.

In [ ]:
# Задание 1. Найдите главные компоненты через сингулярное разложение Z = U·S·Vᵀ (np.linalg.svd, full_matrices=False):
# собственные значения val_svd = S² / (n − 1), доля трёх первых компонент share3. Сравните с val из разбора.
# ваш код

In [ ]:
assert np.allclose(val_svd, val) and abs(share3 - val[:3].sum() / 10) < 1e-12
assert np.allclose(np.abs(Vt[0]), np.abs(vec[:, 0]))
print("Задание 1: верно")

## Задание 2. Сколько компонент и что они значат

80 % дисперсии, связь с ценой.

In [ ]:
# Задание 2. (а) Сколько компонент нужно, чтобы объяснить не менее 80 % дисперсии (m80)? (б) Корреляция первой
# компоненты (scores[:, 0]) с ln базовой цены — r_pc1. (в) Средние значения ГК1 по типам размещения pc1_by_type
# (Series, по возрастанию). Подтверждает ли это название «класс объекта»?
# ваш код

In [ ]:
assert m80 == 5 and r_pc1 > 0.8
assert pc1_by_type.index[0] == "хостел" and pc1_by_type.index[-1] in ("санаторий", "отель")
print("Задание 2: верно")

## Задание 3. Два фактора

Вращение и группировка признаков.

In [ ]:
# Задание 3. Факторный анализ с ДВУМЯ факторами: поверните нагрузки двух первых компонент (varimax) — fa2 (DataFrame
# с колонками "Ф1", "Ф2", "общность"). Для каждого признака — фактор с наибольшей по модулю нагрузкой (factor_of, Series).
# Какие признаки образуют фактор «размер»?
# ваш код

In [ ]:
assert factor_of["rooms"] == factor_of["staff"] != factor_of["room_area"]
assert np.allclose(fa2["общность"].sum(), val[:2].sum()), "вращение не меняет суммарную общность"
print("Задание 3: верно")

## Задание 4. Сегменты k-средних

Профили и названия кластеров.

In [ ]:
# Задание 4. k-средних с k = 4 (функция kmeans): метки lab4, профили prof4 (средние F и медиана price_base по кластеру,
# колонка "объектов"). Дайте кластерам названия (словарь names4: номер -> название), например «премиум», «бюджет».
# ваш код

In [ ]:
assert prof4["объектов"].sum() == 240 and len(prof4) == 4
assert prof4.loc[prof4["цена, медиана"].idxmax(), "stars"] > 3 > prof4.loc[prof4["цена, медиана"].idxmin(), "stars"]
assert inertia4 < kmeans(Z, 3)[2]
print("Задание 4: верно")

## Задание 5. Методы связи

Цепочки одиночной связи.

In [ ]:
# Задание 5. Сравните методы связи: single, complete, average, ward (linkage), разрез на 5 кластеров. Для каждого —
# размеры кластеров (sizes, словарь метод -> отсортированный по убыванию список) и силуэт (sil, словарь).
# Какой метод даёт «цепочку» — один огромный кластер?
# ваш код

In [ ]:
assert sizes["single"][0] > 200, "одиночная связь даёт цепочку"
assert max(sizes["ward"]) < 120 and all(sum(v) == 240 for v in sizes.values())
print("Задание 5: верно")

## Задание 6. LDA: обучение и тест

Точность и полнота по классам.

In [ ]:
# Задание 6. Разделите объекты на обучающие (70 %) и тестовые (30 %): perm = np.random.default_rng(5).permutation(240),
# первые 168 — обучение. Обучите LDA, найдите точность на тесте acc_test и на обучении acc_train, полноту по классам
# recall (Series: доля верно распознанных объектов каждого типа на тесте).
# ваш код

In [ ]:
assert len(te) == 72 and acc_test > 0.7 and acc_train > 0.7
assert set(recall.index) <= set(np.unique(y))
print("Задание 6: верно")

## Задание 7. Выбор k соседей

Кросс-валидация.

In [ ]:
# Задание 7. Подберите число соседей k для knn_predict из [1, 3, 5, 7, 9, 15] по точности на кросс-валидации
# (cv_accuracy): словарь acc_k, лучшее best_k. Сравните с LDA (acc_lda).
# ваш код

In [ ]:
assert set(acc_k) == {1, 3, 5, 7, 9, 15} and acc_k[best_k] == max(acc_k.values())
assert all(0.5 < v < 1 for v in acc_k.values())
print("Задание 7: верно")

## Задание 8 ★. Мини-исследование: сегменты бронирований

Мини-пакетный k-средних на 60 000 строк.

In [ ]:
# Задание 8 ★ (мини-исследование). Сегментация 60 000 бронирований. Признаки: ln цены за ночь, ln ночей, гостей,
# ln(срок + 1). (1) Средние и ст. отклонения — по блокам (суммы x и x²); стандартизуйте (Zb). (2) Мини-пакетный k-средних,
# k = 4: начальные центры — 4 случайных строки (rng = np.random.default_rng(0)), 300 итераций по пакетам из 1000 строк,
# центр j сдвигается к среднему пакета с шагом η = (число точек j в пакете) / (всего точек j до сих пор).
# (3) Метки segs для всех строк, профили segments: размер, медианы признаков, доля агрегатора, доля отмен.
# Сравните инерцию с kmeans2 по всем данным (ratio). Вывод 5–7 предложений: какие сегменты гостей, чем отличаются.
# ваш код

In [ ]:
assert len(segs) == 60_000 and segments["бронирований"].sum() == 60_000
assert np.allclose(mu8, bfeats(BOOKINGS).mean(0)) and ratio < 1.15
print("Задание 8: сегменты построены; вывод проверит преподаватель")

---

## Что вы теперь умеете

| Задача | Инструмент |
|---|---|
| стандартизация | z-оценки |
| главные компоненты | `eigh`, `svd`, нагрузки, доля дисперсии |
| факторный анализ | варимакс, общности |
| кластеры | `kmeans2`, `linkage`, силуэт, ARI, профили |
| классификация | LDA, k ближайших соседей, кросс-валидация |
| большие данные | суммы по блокам, мини-пакетный k-средних |

**Это последний практикум курса.** Дальше — итоговое повторение: вопросы к экзамену и итоговый
тест в итоговом разделе.